<a href="https://colab.research.google.com/github/GouseMohid/Gouse-Innolift-projects/blob/main/Day%2010.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Task 1: Consolidate and Name All Final .pkl Files

First, we'll simulate a machine learning pipeline to create the necessary `.pkl` files: a TF-IDF vectorizer, a label encoder, and a simple classification model. We'll use the `amazon_reviews.csv` dataset for this purpose, assuming a sentiment analysis task.

In [18]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
import joblib
import os

# Ensure a models directory exists
if not os.path.exists('models'):
    os.makedirs('models')

# Load the dataset
df = pd.read_csv('/content/amazon_reviews_10_percent_nan_score.csv')

print(f"Initial DataFrame shape: {df.shape}")
print(f"NaNs in 'Text' before dropna: {df['Text'].isnull().sum()}")
print(f"Unique values in 'Score' column before processing: {df['Score'].unique()}\n")

# Drop rows with NaN in 'Text' as these are critical for NLP
df.dropna(subset=['Text'], inplace=True)

# Convert 'Score' column: map 'Positive' to 1, 'Negative' to 0
# Rows with 'Score' values other than 'Positive' or 'Negative' will become NaN, then dropped.
df['sentiment'] = df['Score'].map({'Positive': 1, 'Negative': 0})

print(f"NaNs in 'sentiment' after mapping: {df['sentiment'].isnull().sum()}")

# Drop rows where sentiment is NaN (i.e., 'Score' was not 'Positive' or 'Negative')
df.dropna(subset=['sentiment'], inplace=True)

# Convert sentiment column to integer after dropping NaNs
df['sentiment'] = df['sentiment'].astype(int)

print(f"DataFrame shape after dropping NaNs in 'Text' or 'sentiment': {df.shape}")

# Check if DataFrame is empty after initial cleaning
if df.empty:
    raise ValueError("DataFrame is empty after initial cleaning (dropping NaNs in 'Text' or invalid 'Score' values). This means your dataset might not contain enough valid data to proceed with classification. Please inspect the dataset.")

X = df['Text']
y = df['sentiment']

# Add a check to ensure at least two classes exist after preprocessing
if y.nunique() < 2:
    # If we still don't have two unique classes, it means the entire dataset
    # is skewed towards one sentiment.
    raise ValueError(f"Not enough classes in the target variable after preprocessing. Found {y.nunique()} unique classes. The dataset is too skewed towards one sentiment, or does not contain enough diverse reviews. Please check the distribution of 'Score' in your original data.")

# Split data (optional, but good practice)
# Use stratify=y to ensure both classes are represented in train and test sets, if possible
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 1. TF-IDF Vectorizer
tfidf_vectorizer = TfidfVectorizer(max_features=5000) # Limit features for simplicity
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)

# 2. Label Encoder
label_encoder = LabelEncoder()
# Fit LabelEncoder on the unique values of y to ensure it knows all possible classes (0 and 1)
label_encoder.fit(y.unique()) # y.unique() should be [0, 1] if y.nunique() is 2
y_train_encoded = label_encoder.transform(y_train)

# 3. Simple Model (Logistic Regression)
model = LogisticRegression(max_iter=1000) # Increase max_iter for convergence
model.fit(X_train_tfidf, y_train_encoded)

print("Model training complete. Now saving artifacts.")

Initial DataFrame shape: (10000, 11)
NaNs in 'Text' before dropna: 0
Unique values in 'Score' column before processing: ['Positive' 'Negative' nan]

NaNs in 'sentiment' after mapping: 1000
DataFrame shape after dropping NaNs in 'Text' or 'sentiment': (9000, 12)
Model training complete. Now saving artifacts.


### Saving .pkl Files

Now we save the trained `TfidfVectorizer`, `LabelEncoder`, and `LogisticRegression` model using `joblib` into a `models/` directory, following the specified naming convention.

In [19]:
# Save your final tuned model
joblib.dump(model, 'models/final_model.pkl')

# Save your label encoder
joblib.dump(label_encoder, 'models/label_encoder.pkl')

# Save your TF-IDF vectorizer
joblib.dump(tfidf_vectorizer, 'models/tfidf_vectorizer.pkl')

# For demonstration of `scaler.pkl`, let's create a dummy one if no numerical features were scaled.
# In a real scenario, this would be fitted on numerical features.
class DummyScaler:
    def transform(self, X): return X
    def inverse_transform(self, X): return X
joblib.dump(DummyScaler(), 'models/scaler.pkl') # Dummy scaler for completeness

print("All .pkl files saved in the 'models/' directory.")

All .pkl files saved in the 'models/' directory.


### Verify every file loads correctly

Let's load each saved `.pkl` file to ensure they are not corrupted and can be successfully reloaded by a Flask application.

In [20]:
import joblib
import numpy as np

# Load each file fresh — as if you are the Flask developer
loaded_model = joblib.load('models/final_model.pkl')
loaded_label_encoder = joblib.load('models/label_encoder.pkl')
loaded_tfidf_vectorizer = joblib.load('models/tfidf_vectorizer.pkl')
loaded_scaler = joblib.load('models/scaler.pkl') # Load the dummy scaler

print("All .pkl files loaded successfully.\n")

# Quick sanity check — make one prediction to confirm the model loaded correctly
# Create a sample input that resembles a text review
sample_review_text = "This product is absolutely amazing and works perfectly!"

# Preprocess the sample input using the loaded vectorizer
sample_review_tfidf = loaded_tfidf_vectorizer.transform([sample_review_text])

# Make a prediction with the loaded model
sample_prediction_encoded = loaded_model.predict(sample_review_tfidf)[0]

# Decode the prediction using the loaded label encoder
sample_prediction_label = loaded_label_encoder.inverse_transform([sample_prediction_encoded])[0]

# Get prediction probabilities
sample_probabilities = loaded_model.predict_proba(sample_review_tfidf)[0]
confidence = max(sample_probabilities) * 100

# Map numerical labels (0, 1) to descriptive strings like 'Negative', 'Positive'
# Assuming 0 is 'Negative' and 1 is 'Positive' based on previous setup
human_readable_label = "Positive" if sample_prediction_label == 1 else "Negative"

print(f"Sample review: '{sample_review_text}'")
print(f"Load test passed. Sample prediction: '{human_readable_label}' with confidence: {confidence:.2f}%")

All .pkl files loaded successfully.

Sample review: 'This product is absolutely amazing and works perfectly!'
Load test passed. Sample prediction: 'Positive' with confidence: 95.09%


### Handoff Checklist

This checklist provides the Flask developer with all the necessary information about the model artifacts.

## Files required for prediction

| File | Purpose |
|---|---|
| final_model.pkl | Trained Logistic Regression classifier for sentiment analysis |
| label_encoder.pkl | LabelEncoder fitted on target `sentiment` |
| tfidf_vectorizer.pkl | TF-IDF vectorizer fitted on `Text` |
| scaler.pkl | DummyScaler (not used in this text classification example) |

## Input columns (for `predict` function input dictionary)
`Text` (string)

## Output
`{ "prediction": "Positive" or "Negative", "confidence": "84.7%" }`

## Task 2 — Write and Test a Single `predict(inputs)` Function

Now, let's implement the `predict` function as specified, which will encapsulate the model loading and prediction logic.

In [23]:
import joblib
import pandas as pd
import numpy as np

def predict(inputs: dict) -> dict:
    """
    Takes a raw input dictionary (exactly what the Flask form will send)
    and returns a dictionary with the prediction and confidence.

    Parameters
    ----------
    inputs : dict
        Raw values from the user, e.g.
        {'review_text': 'This product is great!'}

    Returns
    -------
    dict
        e.g. {'prediction': 'Positive', 'confidence': '84.7%'}
    """

    # Step 1 — Load all required .pkl files
    # Note: These are loaded inside the function to ensure it's self-contained
    # and can be called independently without relying on global variables.
    try:
        model = joblib.load('models/final_model.pkl')
        tfidf_vectorizer = joblib.load('models/tfidf_vectorizer.pkl')
        label_encoder = joblib.load('models/label_encoder.pkl')
        # scaler = joblib.load('models/scaler.pkl') # Uncomment if you used a scaler for numerical features
    except FileNotFoundError as e:
        return {'prediction': 'Error', 'confidence': 'N/A', 'error': f"Missing model file: {e}"}

    # Step 2 — Prepare the input
    # For text classification, we expect 'review_text' in the input dict
    if 'review_text' not in inputs:
        return {'prediction': 'Error', 'confidence': 'N/A', 'error': "Input dictionary must contain 'review_text' key."}

    review_text = inputs['review_text']

    # Step 3 — Apply preprocessing in the EXACT same order as training
    # Vectorize the input text
    processed_input = tfidf_vectorizer.transform([review_text])

    # Step 4 — Make the prediction
    prediction_encoded = model.predict(processed_input)[0]
    probabilities = model.predict_proba(processed_input)[0]

    # Step 5 — Return a clean, human-readable dict
    # Decode the prediction using the label encoder
    class_labels = label_encoder.inverse_transform([prediction_encoded])[0] # Get the actual label

    # Map numerical labels (0, 1) to descriptive strings like 'Negative', 'Positive'
    # Assuming 0 is 'Negative' and 1 is 'Positive' based on previous setup
    human_readable_label = "Positive" if class_labels == 1 else "Negative"

    return {
        'prediction': human_readable_label,
        'confidence': f"{max(probabilities) * 100:.1f}%"
    }

### Test the function with one quick call

In [22]:
sample = {
    'Text': 'This product is absolutely amazing and works perfectly! I love it.'
}

result = predict(sample)
print(result)

sample_negative = {
    'Text': 'This product is terrible and broke after one use. Very disappointed.'
}

result_negative = predict(sample_negative)
print(result_negative)

{'prediction': 'Positive', 'confidence': '97.8%'}
{'prediction': 'Negative', 'confidence': '75.4%'}


## Task 3 — Run 10 Test Cases Through `predict()` and Log Results

Now we will run 10 test cases through the `predict` function, including normal, high-risk, edge, known, and invalid inputs, and log the results in a table.

In [25]:
import pandas as pd

# Define your 10 test cases as a list of dicts
test_cases = [
    # Case 1 — Normal / low risk (likely negative sentiment)
    {'Text': 'This product is okay, not great but not bad either.'},

    # Case 2 — Normal (likely positive sentiment)
    {'Text': 'Absolutely fantastic product! Highly recommend it to everyone.'},

    # Case 3 — Normal / average (neutral or slightly positive/negative)
    {'Text': 'It works as advertised. Nothing special to note.'},

    # Case 4 — High-risk (very negative sentiment)
    {'Text': 'This is the worst product I have ever bought. A complete scam.'},

    # Case 5 — High-risk (another very negative sentiment)
    {'Text': 'Utter garbage. Do not waste your money on this trash.'},

    # Case 6 — Edge case: minimum valid input (very short review)
    {'Text': 'Good.'},

    # Case 7 — Edge case: maximum valid input (very long review, will be truncated by TF-IDF if needed)
    {'Text': 'This product exceeded my expectations in every possible way. From the moment I unboxed it, I was impressed by its sleek design and robust build quality. Setting it up was a breeze, taking only a few minutes thanks to the clear instructions. The performance is outstanding, handling all my tasks with ease and efficiency. The battery life is surprisingly long, allowing me to use it for extended periods without needing a recharge. I particularly appreciate the thoughtful features and the intuitive user interface. This is truly a game-changer and has made my daily routine much more enjoyable. I\'ve recommended it to all my friends and family, and they\'ve all been equally delighted. Definitely a five-star purchase! The customer support is also very responsive and helpful, though I barely needed them because the product works flawlessly. I will certainly be buying more from this brand in the future. Highly satisfied and a loyal customer now!'},

    # Case 8 — Known test row (example of input that should yield negative)
    {'Text': 'I am so disappointed with this item. It broke after one use.'},

    # Case 9 — Known test row (example of input that should yield positive)
    {'Text': 'The quality is superb for the price. Very happy with my purchase.'},

    # Case 10 — Invalid input (missing field, should return error message)
    {'invalid_key': 'This is an invalid input example.'},

    # Case 11 — Invalid input (non-string value for review_text)
    {'Text': 12345} # Non-string review_text
]

# Run all and log results
results = []
for i, case in enumerate(test_cases, 1):
    result = predict(case)
    results.append({
        'Case': i,
        'Prediction': result.get('prediction', 'Error'),
        'Confidence': result.get('confidence', 'N/A'),
        'Status': 'PASS' if result.get('prediction') != 'Error' else 'ERROR',
        'Error_Message': result.get('error', '') # Log error message for debugging
    })

log_df = pd.DataFrame(results)
# Print the DataFrame without index for cleaner output
print(log_df.to_string(index=False))


 Case Prediction Confidence Status                                                   Error_Message
    1   Negative      77.6%   PASS                                                                
    2   Positive      96.2%   PASS                                                                
    3   Positive      84.1%   PASS                                                                
    4   Negative      64.7%   PASS                                                                
    5   Negative      83.7%   PASS                                                                
    6   Positive      99.1%   PASS                                                                
    7   Positive      90.4%   PASS                                                                
    8   Negative      53.1%   PASS                                                                
    9   Positive      95.8%   PASS                                                                
   10     

## Task 4 — Write the Model Summary Card

This section provides a summary card for the developed model, detailing its purpose, algorithm, performance, and usage.

## Model Summary Card

### Project
Sentiment Analysis for Amazon Reviews · E-commerce / Customer Feedback

### Algorithm
Logistic Regression (with TF-IDF Vectorization)

### Dataset
Amazon Fine Food Reviews · 10,000 rows (subset) · `Text` feature, `Score` for sentiment

### Final Performance
| Metric | Score |
|---|---|
| Accuracy | _(to be filled after full evaluation)_ |
| F1-Score (weighted) | _(to be filled after full evaluation)_ |
| Cross-validation mean | _(to be filled after full evaluation)_ |

### Input Features (for `predict` function)
| Column | Type | Example value |
|---|---|---|
| `Text` | string | "This product is great!" |

### Required .pkl Files
| File | Contents |
|---|---|
| `final_model.pkl` | Trained Logistic Regression classifier |
| `label_encoder.pkl` | LabelEncoder fitted on target sentiment (0: Negative, 1: Positive) |
| `tfidf_vectorizer.pkl` | TF-IDF Vectorizer fitted on `Text` |
| `scaler.pkl` | DummyScaler (not used in this text classification example) |

### Sample Input
```python
{
    'Text': 'This is a fantastic product, I really enjoyed using it!'
}
```

### Sample Output
```python
{
    'prediction': 'Positive',
    'confidence': '92.5%'
}
```

### How to use
```python
import joblib
result = predict(your_input_dict)
```

In [16]:
import pandas as pd

# Load the dataset to inspect it
inspection_df = pd.read_csv('/content/amazon_reviews.csv')

# Display the first 10 rows
display(inspection_df.head(10))

,Unnamed: 0,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,Positive,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,Negative,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,Positive,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...
3,3,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3,Negative,1307923200,Cough Medicine,If you are looking for the secret ingredient i...
4,4,5,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0,0,Positive,1350777600,Great taffy,Great taffy at a great price. There was a wid...
5,5,6,B006K2ZZ7K,ADT0SRK1MGOEU,Twoapennything,0,0,Positive,1342051200,Nice Taffy,I got a wild hair for taffy and ordered this f...
6,6,7,B006K2ZZ7K,A1SP2KVKFXXRU1,David C. Sullivan,0,0,Positive,1340150400,Great! Just as good as the expensive brands!,This saltwater taffy had great flavors and was...
7,7,8,B006K2ZZ7K,A3JRGQVEQN31IQ,Pamela G. Williams,0,0,Positive,1336003200,"Wonderful, tasty taffy",This taffy is so good. It is very soft and ch...
8,8,9,B000E7L2R4,A1MZYO9TZK0BBI,R. James,1,1,Positive,1322006400,Yay Barley,Right now I'm mostly just sprouting this so my...
9,9,10,B00171APVA,A21BT40VZCCYT4,Carol A. Reed,0,0,Positive,1351209600,Healthy Dog Food,This is a very healthy dog food. Good for thei...


In [26]:
import joblib
import pandas as pd
import numpy as np
import os # Import os for path manipulation

def predict(inputs: dict) -> dict:
    """
    Takes a raw input dictionary (exactly what the Flask form will send)
    and returns a dictionary with the prediction and confidence.

    Parameters
    ----------
    inputs : dict
        Raw values from the user, e.g.
        {'Text': 'This product is great!'}

    Returns
    -------
    dict
        e.g. {'prediction': 'Positive', 'confidence': '84.7%'}
    """

    try:
        # Define the base path for model files
        model_base_path = '/content/models/' # Use absolute path

        # Step 1 — Load all required .pkl files
        # Note: These are loaded inside the function to ensure it's self-contained
        # and can be called independently without relying on global variables.
        model = joblib.load(os.path.join(model_base_path, 'final_model.pkl'))
        tfidf_vectorizer = joblib.load(os.path.join(model_base_path, 'tfidf_vectorizer.pkl'))
        label_encoder = joblib.load(os.path.join(model_base_path, 'label_encoder.pkl'))
        # scaler = joblib.load(os.path.join(model_base_path, 'scaler.pkl')) # Uncomment if you used a scaler for numerical features

        # Step 2 — Prepare the input
        # For text classification, we expect 'Text' in the input dict
        if 'Text' not in inputs or not isinstance(inputs['Text'], str):
            raise ValueError("Input dictionary must contain a 'Text' key with a string value.")

        review_text = inputs['Text']

        # Step 3 — Apply preprocessing in the EXACT same order as training
        # Vectorize the input text
        processed_input = tfidf_vectorizer.transform([review_text])

        # Step 4 — Make the prediction
        prediction_encoded = model.predict(processed_input)[0]
        probabilities = model.predict_proba(processed_input)[0]

        # Step 5 — Return a clean, human-readable dict
        # Decode the prediction using the label encoder
        if not hasattr(label_encoder, 'classes_'):
            raise RuntimeError("LabelEncoder has not been fitted or is corrupted.")

        original_label_value = label_encoder.inverse_transform([prediction_encoded])[0]

        class_mapping = {0: 'Negative', 1: 'Positive'}
        human_readable_label = class_mapping.get(original_label_value, 'Unknown')

        return {
            'prediction': human_readable_label,
            'confidence': f"{max(probabilities) * 100:.1f}%"
        }
    except FileNotFoundError as e:
        return {'prediction': 'Error', 'confidence': 'N/A', 'error': f"Missing model file: {e}"}
    except ValueError as e:
        return {'prediction': 'Error', 'confidence': 'N/A', 'error': str(e)}
    except Exception as e:
        return {'prediction': 'Error', 'confidence': 'N/A', 'error': f"An unexpected error occurred: {e}"}
